# DeepFM for Search Relevance Ranking
### MS MARCO Passage Ranking — ML1 Final Project
**University of Chicago, MSc in Applied Data Science**

---

## 1. Model Description & Applications

**DeepFM** (Deep Factorization Machine) is a neural architecture that jointly trains a Factorization Machine component and a Deep Neural Network component over shared input embeddings. It was introduced by Huawei Noah's Ark Lab (Guo et al., 2017) specifically to address the challenge of learning both low- and high-order feature interactions without manual feature engineering.

The core insight is that click-through rate prediction, relevance ranking, and recommendation all share a common structure: the signal is determined not by individual features alone, but by *interactions* between features. DeepFM learns these interactions end-to-end.

### Real-World Applications
| Company | Use Case | Reference |
|---------|----------|----------|
| Huawei | App store recommendation (CTR prediction) | [Guo et al., 2017](https://arxiv.org/abs/1703.04247) |
| Alibaba | Display advertising ranking | [Zhou et al., 2018](https://arxiv.org/abs/1809.03672) |
| Criteo | Ad click prediction (Criteo benchmark) | [Criteo Dataset](https://www.kaggle.com/c/criteo-display-ad-challenge) |
| Microsoft | Bing multi-stage ranking stack | [MS MARCO Benchmark](https://microsoft.github.io/msmarco/) |
| JD.com | E-commerce product ranking | [Lian et al., 2018](https://arxiv.org/abs/1803.05170) |

---
## 2. Key Components & Mathematical Formulation

### 2.1 Input Representation

The input to DeepFM is a feature vector $\mathbf{x} \in \mathbb{R}^d$ where $d$ is the total number of features. Features can be dense (continuous) or sparse (categorical, one-hot encoded). Each feature field $i$ is associated with a learned embedding vector $\mathbf{v}_i \in \mathbb{R}^k$, where $k$ is the embedding dimension (typically 4–16).

$$\mathbf{x} = [x_1, x_2, \ldots, x_d]$$

---

### 2.2 Factorization Machine (FM) Component

The FM component captures **first-order** and **second-order** (pairwise) feature interactions.

**First-order term** — linear contribution of each feature:
$$y_{\text{order1}} = w_0 + \sum_{i=1}^{d} w_i x_i$$

where $w_0$ is a global bias and $w_i$ are per-feature weights.

**Second-order term** — pairwise interactions via factorized weights:
$$y_{\text{order2}} = \sum_{i=1}^{d} \sum_{j=i+1}^{d} \langle \mathbf{v}_i, \mathbf{v}_j \rangle \cdot x_i x_j$$

where $\langle \mathbf{v}_i, \mathbf{v}_j \rangle = \sum_{l=1}^{k} v_{il} \cdot v_{jl}$ is the dot product of embedding vectors.

This is computed efficiently in $O(kd)$ using the identity:
$$y_{\text{order2}} = \frac{1}{2} \sum_{l=1}^{k} \left[ \left( \sum_{i=1}^{d} v_{il} x_i \right)^2 - \sum_{i=1}^{d} v_{il}^2 x_i^2 \right]$$

**Combined FM output:**
$$y_{\text{FM}} = y_{\text{order1}} + y_{\text{order2}}$$

---

### 2.3 Deep Neural Network (DNN) Component

The DNN component captures **high-order** and **non-linear** feature interactions. It takes the concatenation of all feature embeddings as input:

$$\mathbf{a}^{(0)} = [\mathbf{v}_1 \cdot x_1, \ \mathbf{v}_2 \cdot x_2, \ \ldots, \ \mathbf{v}_d \cdot x_d]$$

Each hidden layer applies:
$$\mathbf{a}^{(l)} = \text{ReLU}\left( \mathbf{W}^{(l)} \mathbf{a}^{(l-1)} + \mathbf{b}^{(l)} \right)$$

where $\mathbf{W}^{(l)}$ and $\mathbf{b}^{(l)}$ are the weight matrix and bias of layer $l$.

The DNN output is a scalar:
$$y_{\text{DNN}} = \mathbf{W}^{\text{out}} \cdot \mathbf{a}^{(L)} + b^{\text{out}}$$

---

### 2.4 DeepFM Output

The FM and DNN components share the same embedding layer. Their outputs are summed and passed through a sigmoid to produce a relevance probability:

$$\hat{y} = \sigma\left( y_{\text{FM}} + y_{\text{DNN}} \right) = \frac{1}{1 + e^{-(y_{\text{FM}} + y_{\text{DNN}})}}$$

**Training Objective** — binary cross-entropy loss over labeled query-passage pairs:
$$\mathcal{L} = -\frac{1}{N} \sum_{i=1}^{N} \left[ y_i \log \hat{y}_i + (1 - y_i) \log(1 - \hat{y}_i) \right]$$

where $y_i \in \{0, 1\}$ is the binary relevance label.

---

### 2.5 Architecture Diagram

```
Input Features:  [BM25, q_len, p_len, term_overlap, exact_match, ...]
                              |
                    [Embedding Layer]  <--- shared embeddings
                    /              \
              [FM Layer]        [DNN Layers]
          (order1 + order2)    (ReLU activations)
                    \              /
                     [Sum + Sigmoid]
                           |
                  ŷ ∈ [0,1] (relevance score)
```

---
## 3. Algorithm Description

### Training Algorithm

**Input:** Training set $\{(\mathbf{x}_i, y_i)\}_{i=1}^N$, learning rate $\eta$, embedding dim $k$, DNN layers $[h_1, h_2, \ldots]$

**Algorithm:**
1. Initialize embedding vectors $\mathbf{v}_i \sim \mathcal{N}(0, 0.01)$ for all features
2. Initialize DNN weights $\mathbf{W}^{(l)}$ via Xavier initialization
3. **For** each mini-batch $\mathcal{B}$:
   - a. Compute $y_{\text{FM}}$ via first + second order terms
   - b. Compute $y_{\text{DNN}}$ via forward pass through hidden layers
   - c. Compute $\hat{y} = \sigma(y_{\text{FM}} + y_{\text{DNN}})$
   - d. Compute loss $\mathcal{L}$
   - e. Backpropagate gradients through both components (shared embedding receives gradients from both paths)
   - f. Update all parameters via Adam: $\theta \leftarrow \theta - \eta \cdot \hat{m} / (\sqrt{\hat{v}} + \epsilon)$
4. Evaluate on validation set after each epoch
5. Early stop if validation loss does not improve for $p$ epochs

**Inference (Ranking):**
1. For a query $q$ and candidate passages $\{p_1, \ldots, p_K\}$
2. Extract features $\mathbf{x}_{q,p_j}$ for each pair
3. Compute $\hat{y}_{q,p_j} = \text{DeepFM}(\mathbf{x}_{q,p_j})$
4. Return passages sorted by $\hat{y}$ in descending order

### Evaluation Metrics

**NDCG@10** (Normalized Discounted Cumulative Gain):
$$\text{NDCG@}K = \frac{\text{DCG@}K}{\text{IDCG@}K}, \quad \text{DCG@}K = \sum_{i=1}^{K} \frac{\text{rel}_i}{\log_2(i+1)}$$

**MRR** (Mean Reciprocal Rank):
$$\text{MRR} = \frac{1}{|Q|} \sum_{q \in Q} \frac{1}{\text{rank}_q}$$

where $\text{rank}_q$ is the position of the first relevant passage for query $q$.

---
## 4. Package Overview & Selection

| Package | Key Methods | Pros | Cons |
|---------|------------|------|------|
| `deepctr-torch` | `DeepFM`, `FM`, `xDeepFM`, `AutoInt` | Rich model zoo, clean API, actively maintained | Requires PyTorch |
| `xlearn` | `create_fm`, `create_ffm` | Very fast (C++ backend), memory efficient | Less flexible, harder to customize |
| `cuML` | `FactorizationMachine` | GPU-accelerated, scikit-learn API | NVIDIA GPU required, limited to basic FM |
| `torchfm` | `FMModel`, `DeepFMModel` | Minimal, educational | Less production-ready |

**Selected Package: `deepctr-torch`**

Reasoning: deepctr-torch provides a production-grade DeepFM implementation with a clean feature column API that mirrors how real search ranking systems define features. It supports both dense and sparse inputs natively, includes dropout and batch normalization, and the codebase is readable enough to understand what's happening under the hood — which matters for the mathematical formulation section of this presentation.

---
## 5. Feature Engineering — MS MARCO

This section builds the full feature pipeline from raw MS MARCO data to a model-ready DataFrame.

In [ ]:
# ─── Environment setup ────────────────────────────────────────────────────────
# !pip install rank_bm25 deepctr-torch torch pandas numpy scikit-learn tqdm ir_datasets

import warnings
warnings.filterwarnings('ignore')

import re
import math
import numpy as np
import pandas as pd
from tqdm import tqdm
from collections import Counter
from rank_bm25 import BM25Okapi

In [ ]:
# ─── 5.1  Load MS MARCO dev-small via ir_datasets ─────────────────────────────
# ir_datasets provides a clean interface to MS MARCO without manual downloads.
# We use the passage ranking dev set: ~6,980 queries, each with BM25-retrieved 
# candidate passages and binary relevance labels.

import ir_datasets

dataset = ir_datasets.load("msmarco-passage/dev/small")

# Build passage lookup: pid -> text
print("Loading passages...")
passages = {doc.doc_id: doc.text for doc in tqdm(dataset.docs_iter())}
print(f"Loaded {len(passages):,} passages")

# Build query lookup: qid -> text
queries = {q.query_id: q.text for q in dataset.queries_iter()}
print(f"Loaded {len(queries):,} queries")

In [ ]:
# ─── 5.2  Build relevance labels from qrels ───────────────────────────────────
# MS MARCO uses binary relevance: 1 = relevant, 0 = not relevant.
# qrels maps (query_id, doc_id) -> relevance grade.

qrels = {}  # {qid: set of relevant pids}
for qrel in dataset.qrels_iter():
    if qrel.relevance > 0:
        qrels.setdefault(qrel.query_id, set()).add(qrel.doc_id)

print(f"Queries with at least one relevant passage: {len(qrels):,}")

In [ ]:
# ─── 5.3  BM25 Retrieval — first stage (top-100 candidates per query) ─────────
# In production search, BM25 is the recall stage: cheap lexical matching that
# generates a candidate set. DeepFM then re-ranks this candidate set.
# 
# For efficiency we work with a sample of queries. Adjust SAMPLE_N as needed.

SAMPLE_N = 500   # number of queries to use; increase for final submission
TOP_K    = 50    # candidates per query from BM25

sampled_qids = list(qrels.keys())[:SAMPLE_N]

def simple_tokenize(text: str) -> list:
    """Lowercase + split on non-alphanumeric characters."""
    return re.sub(r'[^a-z0-9\s]', '', text.lower()).split()

# Build a BM25 index over all passages (or a large subset)
# NOTE: indexing all 8.8M passages is slow — for dev purposes index a subset
# that covers at least all passages associated with our sampled queries.
print("Collecting relevant pids for sampled queries...")
relevant_pids = set()
for qid in sampled_qids:
    relevant_pids.update(qrels.get(qid, set()))

# For BM25 retrieval we need a corpus. We'll use all passages (memory permitting)
# or a filtered subset. Here we use all for correctness.
print("Building BM25 index (this may take a few minutes)...")
pid_list    = list(passages.keys())
corpus_tok  = [simple_tokenize(passages[pid]) for pid in tqdm(pid_list)]
bm25_index  = BM25Okapi(corpus_tok)
pid_to_idx  = {pid: i for i, pid in enumerate(pid_list)}
print("BM25 index ready.")

In [ ]:
# ─── 5.4  Feature Engineering ─────────────────────────────────────────────────
# We extract 10 features for each (query, passage) pair. 
# These cover lexical overlap, length signals, and retrieval scores.
#
# Feature set:
#   1.  bm25_score         — BM25 retrieval score (primary lexical signal)
#   2.  bm25_rank          — rank position in BM25 results (1 = top)
#   3.  query_len          — number of tokens in query
#   4.  passage_len        — number of tokens in passage
#   5.  query_passage_ratio — query_len / passage_len (length mismatch)
#   6.  term_overlap_count — # of query tokens found in passage
#   7.  term_overlap_frac  — term_overlap_count / query_len (coverage)
#   8.  exact_phrase_match — 1 if full query string appears in passage
#   9.  idf_weighted_overlap — sum of IDF weights for matched query terms
#  10.  passage_avg_idf    — mean IDF of passage tokens (content richness)

def compute_idf(corpus_tokenized: list) -> dict:
    """Compute IDF scores from a tokenized corpus."""
    N = len(corpus_tokenized)
    df = Counter()
    for doc in corpus_tokenized:
        for term in set(doc):
            df[term] += 1
    return {term: math.log((N - freq + 0.5) / (freq + 0.5) + 1)
            for term, freq in df.items()}

print("Computing IDF scores...")
idf_scores = compute_idf(corpus_tok)
print(f"Vocabulary size: {len(idf_scores):,}")

In [ ]:
def extract_features(query: str, passage: str, bm25_score: float, 
                     bm25_rank: int, idf: dict) -> dict:
    """
    Extract all 10 features for a single (query, passage) pair.
    
    Parameters
    ----------
    query      : raw query string
    passage    : raw passage string
    bm25_score : BM25Okapi score for this pair
    bm25_rank  : rank position of this passage in BM25 results (1-indexed)
    idf        : precomputed IDF dictionary {term: float}
    
    Returns
    -------
    dict of feature_name -> float
    """
    q_tokens = simple_tokenize(query)
    p_tokens = simple_tokenize(passage)
    q_set    = set(q_tokens)
    p_set    = set(p_tokens)
    
    q_len = max(len(q_tokens), 1)  # avoid division by zero
    p_len = max(len(p_tokens), 1)
    
    # Term overlap: query tokens present in passage
    matched_terms       = q_set.intersection(p_set)
    term_overlap_count  = len(matched_terms)
    term_overlap_frac   = term_overlap_count / q_len
    
    # Exact phrase match (case-insensitive)
    exact_match = 1 if query.lower() in passage.lower() else 0
    
    # IDF-weighted overlap: sum IDF of matched query terms
    idf_weighted_overlap = sum(idf.get(t, 0.0) for t in matched_terms)
    
    # Average IDF of passage tokens (proxy for content informativeness)
    passage_avg_idf = np.mean([idf.get(t, 0.0) for t in p_tokens]) if p_tokens else 0.0
    
    return {
        'bm25_score'          : bm25_score,
        'bm25_rank'           : bm25_rank,
        'query_len'           : q_len,
        'passage_len'         : p_len,
        'query_passage_ratio' : q_len / p_len,
        'term_overlap_count'  : term_overlap_count,
        'term_overlap_frac'   : term_overlap_frac,
        'exact_phrase_match'  : exact_match,
        'idf_weighted_overlap': idf_weighted_overlap,
        'passage_avg_idf'     : passage_avg_idf,
    }

In [ ]:
# ─── 5.5  Build training dataset ──────────────────────────────────────────────
# For each sampled query:
#   - retrieve TOP_K candidate passages from BM25
#   - extract features for each (query, passage) pair
#   - assign binary label: 1 if passage is in qrels, else 0
#
# This produces a dataset where each row = one (query, passage) pair.

rows = []

print(f"Building feature dataset for {SAMPLE_N} queries...")
for qid in tqdm(sampled_qids):
    query        = queries[qid]
    query_tokens = simple_tokenize(query)
    rel_pids     = qrels.get(qid, set())
    
    # BM25 scores for this query over entire corpus
    scores      = bm25_index.get_scores(query_tokens)  # array of length = corpus
    top_indices = np.argsort(scores)[::-1][:TOP_K]     # top-K indices
    
    for rank, idx in enumerate(top_indices, start=1):
        pid     = pid_list[idx]
        passage = passages[pid]
        score   = scores[idx]
        label   = 1 if pid in rel_pids else 0
        
        feats = extract_features(
            query    = query,
            passage  = passage,
            bm25_score = score,
            bm25_rank  = rank,
            idf      = idf_scores
        )
        feats['query_id']   = qid
        feats['passage_id'] = pid
        feats['label']      = label
        rows.append(feats)

df = pd.DataFrame(rows)
print(f"\nDataset shape: {df.shape}")
print(f"Positive labels (relevant): {df['label'].sum():,}")
print(f"Negative labels:           {(df['label']==0).sum():,}")
print(f"Positive rate:             {df['label'].mean():.3%}")

In [ ]:
# Preview the feature matrix
df.drop(columns=['query_id', 'passage_id']).describe().round(3)

In [ ]:
# ─── 5.6  Feature Distribution Analysis ───────────────────────────────────────
# Examine whether features differ between relevant and non-relevant passages.
# This is your first sanity check — good features should show separation.

import matplotlib.pyplot as plt

FEATURE_COLS = [
    'bm25_score', 'bm25_rank', 'query_len', 'passage_len',
    'query_passage_ratio', 'term_overlap_count', 'term_overlap_frac',
    'idf_weighted_overlap', 'passage_avg_idf'
]

fig, axes = plt.subplots(3, 3, figsize=(14, 10))
axes = axes.flatten()

for i, feat in enumerate(FEATURE_COLS):
    ax = axes[i]
    df[df['label']==1][feat].hist(bins=40, alpha=0.6, color='steelblue', 
                                   label='Relevant', density=True, ax=ax)
    df[df['label']==0][feat].hist(bins=40, alpha=0.6, color='salmon', 
                                   label='Not Relevant', density=True, ax=ax)
    ax.set_title(feat, fontsize=10)
    ax.legend(fontsize=8)

plt.suptitle('Feature Distributions: Relevant vs Non-Relevant Passages', 
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=120, bbox_inches='tight')
plt.show()
print("\nFeatures with clear separation (higher mean for relevant passages):")
means = df.groupby('label')[FEATURE_COLS].mean()
print(means.T.rename(columns={0: 'Not Relevant', 1: 'Relevant'}).round(3))

In [ ]:
# ─── 5.7  Train/Validation/Test Split ─────────────────────────────────────────
# Split at the QUERY level (not row level) to prevent data leakage.
# A query's passages must all be in the same split.

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

unique_qids = df['query_id'].unique()

# 70% train, 15% val, 15% test
train_qids, temp_qids = train_test_split(unique_qids, test_size=0.30, 
                                          random_state=42)
val_qids,  test_qids  = train_test_split(temp_qids,   test_size=0.50, 
                                          random_state=42)

train_df = df[df['query_id'].isin(train_qids)].reset_index(drop=True)
val_df   = df[df['query_id'].isin(val_qids)].reset_index(drop=True)
test_df  = df[df['query_id'].isin(test_qids)].reset_index(drop=True)

print(f"Train: {len(train_df):,} rows ({len(train_qids):,} queries)")
print(f"Val:   {len(val_df):,} rows ({len(val_qids):,} queries)")
print(f"Test:  {len(test_df):,} rows ({len(test_qids):,} queries)")

# Scale features — fit scaler on TRAINING data only (no leakage)
scaler    = StandardScaler()
DENSE_FEATS = [c for c in FEATURE_COLS if c != 'exact_phrase_match']

train_df[DENSE_FEATS] = scaler.fit_transform(train_df[DENSE_FEATS])
val_df[DENSE_FEATS]   = scaler.transform(val_df[DENSE_FEATS])
test_df[DENSE_FEATS]  = scaler.transform(test_df[DENSE_FEATS])

print("\nScaling complete. Features are zero-mean, unit-variance (training stats).")

In [ ]:
# ─── 5.8  Baseline: BM25 alone ────────────────────────────────────────────────
# Before training DeepFM, establish the BM25 baseline score.
# Our DeepFM must beat this to justify the added complexity.

def ndcg_at_k(group: pd.DataFrame, k: int = 10) -> float:
    """Compute NDCG@K for a single query group."""
    group = group.sort_values('bm25_score', ascending=False).head(k)
    dcg   = sum(rel / math.log2(i + 2) 
                for i, rel in enumerate(group['label']))
    # Ideal DCG: sort by true labels
    ideal = group.sort_values('label', ascending=False)
    idcg  = sum(rel / math.log2(i + 2) 
                for i, rel in enumerate(ideal['label']))
    return dcg / idcg if idcg > 0 else 0.0

def mrr(group: pd.DataFrame, score_col: str = 'bm25_score') -> float:
    """Compute MRR for a single query group."""
    group = group.sort_values(score_col, ascending=False).reset_index(drop=True)
    for i, row in group.iterrows():
        if row['label'] == 1:
            return 1.0 / (i + 1)
    return 0.0

# Compute BM25 baseline on test set
bm25_ndcg = test_df.groupby('query_id').apply(
    lambda g: ndcg_at_k(g, k=10)).mean()
bm25_mrr  = test_df.groupby('query_id').apply(
    lambda g: mrr(g, 'bm25_score')).mean()

print("=" * 40)
print(f"BM25 Baseline  — NDCG@10: {bm25_ndcg:.4f}")
print(f"BM25 Baseline  — MRR:     {bm25_mrr:.4f}")
print("=" * 40)
print("\nThis is our target to beat with DeepFM.")

In [ ]:
# Save processed data for model training section
train_df.to_csv('train_features.csv', index=False)
val_df.to_csv('val_features.csv',     index=False)
test_df.to_csv('test_features.csv',   index=False)
print("Feature files saved. Ready for DeepFM training in Section 6.")

---
## Next Sections (to be completed by team)

- **Section 6:** DeepFM training with `deepctr-torch` (basic example on synthetic data)
- **Section 7:** Real-world application — full MS MARCO training loop, evaluation, sensitivity analysis
- **Section 8:** Requirements (Python version, package versions, OS)
- **Section 9:** Interactive demo (Gradio widget)

---
### References
- Guo et al. (2017). *DeepFM: A Factorization-Machine based Neural Network for CTR Prediction*. [arXiv:1703.04247](https://arxiv.org/abs/1703.04247)
- Bajaj et al. (2018). *MS MARCO: A Human Generated MAchine Reading COmprehension Dataset*. [arXiv:1611.09268](https://arxiv.org/abs/1611.09268)
- deepctr-torch documentation: https://deepctr-torch.readthedocs.io